<a href="https://colab.research.google.com/github/udplabs/okta-ai-poc/blob/feature%2Fagent-registration/colabs/agent_registration_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Okta AI Agent Registration - Complete Flow

This notebook follows the Okta Postman collection flow for registering an AI agent in Okta.

## Complete Registration Flow (10 Steps)

1. **Get Access Token** - Client Credentials + private_key_jwt
2. **Get App ID** - Fetch the linked app
3. **Create Agent** - Register new AI agent
4. **Get Agent ID** - Extract from operation response header
5. **Add Public Key** - Register agent's public JWK
6. **Get Org ID** - Fetch Okta organization ID
7. **Get Owner Group ID** - Retrieve owner group
8. **Assign Owner** - Set owner/group for the agent
9. **Activate Agent** - Transition to ACTIVE status
10. **Add Managed Connection** - Create connection with IDENTITY_ASSERTION_CUSTOM_AS

### Important Training Notes
- This notebook intentionally uses direct HTTP calls instead of an SDK to teach protocol-level behavior.
- Two key materials are used:
  - `REG_SERVICE_PRIVATE_JWK` authenticates the service client for token minting.
  - `PUBLIC_JWK`/`PRIVATE_JWK` generated in Step 0 are the agent credential pair.

---

## Configuration

In [ ]:
OKTA_DOMAIN = 'https://{your_domain}.oktapreview.com' # @param {"type":"string", "placeholder": "Enter Okta domain", "description":"Enter your entire Okta domain (including https)"}
CLIENT_ID = '' # @param {"type":"string","placeholder":"Optional: Enter app client Id"}
APP_NAME = '' # @param {"type":"string", "placeholder": "Enter app name", "description":"Enter the name of the app associated to the agent."}

REG_SERVICE_CLIENT_ID = '' # @param {"type":"string","placeholder":"Enter the service account client Id", "description":"The client Id of the service account to be used to handle interfacing with Okta."}
REG_SERVICE_PRIVATE_JWK = {} # @param {"type":"raw", "placeholder": "Enter the JWK as JSON", "description":"Enter the private JWK of the service account to be used to handle interfacing with Okta."}

AUTHZ_SERVER_ID = '' # @param {"type":"string","placeholder":"Enter the authorization server Id.", "description":"The authorization server Id to be used for the agent's managed connection."}
STRICT_MODE = True # @param {"type":"boolean", "description":"Fail fast on missing prerequisites (for example owner group not found)."}

# Agent Configuration
AGENT_NAME = "Demo AI Agent For Registration Demo" # @param {"type":"string","placeholder":"Enter the name of the agent.", "description":"The name of the agent to be registered.", "default":"Demo AI Agent For Registration Demo"}
AGENT_DESCRIPTION = "A demonstration AI agent for Registration Demo" # @param {"type":"string","placeholder":"Enter the description of the agent.", "description":"The description of the agent to be registered.", "default":"A demonstration AI agent for Registration Demo"}

import re

# Validate OKTA_DOMAIN format
okta_domain_regex = r"^https:\/\/[a-zA-Z0-9-]+\.(okta|oktapreview|okta-emea)\.com$"
if not re.match(okta_domain_regex, OKTA_DOMAIN):
    raise ValueError(
        f"Invalid OKTA_DOMAIN: '{OKTA_DOMAIN}'. "
        "Expected format: 'https://<your-domain>.<okta|oktapreview|okta-emea>.com'"
    )

# Validate other required variables
if not APP_NAME:
    raise ValueError("APP_NAME cannot be empty.")
if not REG_SERVICE_CLIENT_ID:
    raise ValueError("REG_SERVICE_CLIENT_ID cannot be empty.")
if not REG_SERVICE_PRIVATE_JWK:
    raise ValueError("REG_SERVICE_PRIVATE_JWK cannot be empty.")
if not AUTHZ_SERVER_ID:
    raise ValueError("AUTHZ_SERVER_ID cannot be empty.")
if not AGENT_NAME:
    raise ValueError("AGENT_NAME cannot be empty.")
if not AGENT_DESCRIPTION:
    raise ValueError("AGENT_DESCRIPTION cannot be empty.")

print("Configuration loaded!")
print(f"Domain: {OKTA_DOMAIN}")
print(f"Authz Server ID: {AUTHZ_SERVER_ID}")
print(f"Strict mode: {STRICT_MODE}")

## Helper Functions

In [ ]:
import json
import requests
import base64
import time
import uuid
import jwt
from typing import Dict, Any, Optional, Tuple
from urllib.parse import urlparse
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend

def get_okta_partition(okta_domain: str) -> str:
    """Infer ORN partition from Okta domain."""
    hostname = urlparse(okta_domain).hostname or ""
    if hostname.endswith(".oktapreview.com"):
        return "oktapreview"
    if hostname.endswith(".okta-emea.com"):
        return "okta-emea"
    if hostname.endswith(".okta.com"):
        return "okta"
    raise ValueError(f"Unsupported Okta domain for ORN partition derivation: {okta_domain}")

def _generate_jwt_assertion_for_service_token(
        client_id: str,
        audience: str,
        private_jwk: Dict[str, Any]
    ) -> str:
        """
        Generate a JWT assertion for client authentication using private JWK
        Based on RFC 7523 (JSON Web Token (JWT) Profile for OAuth 2.0 Client Authentication)
        """

        if private_jwk is None or not isinstance(private_jwk, dict):
            raise ValueError("private_jwk must be a non-empty dict.")

        required_base_fields = ("n", "e", "d")
        missing_base_fields = [field for field in required_base_fields if not private_jwk.get(field)]
        if missing_base_fields:
            raise ValueError(
                f"private_jwk is missing required fields: {', '.join(missing_base_fields)}"
            )

        # Get current time
        now = int(time.time())

        # Prepare header
        header = {
            "kid": private_jwk.get("kid"),
            "alg": "RS256"
        }

        # Prepare payload
        payload = {
            "iss": client_id,
            "aud": audience,
            "sub": client_id,
            "exp": now + 60,  # Expires in 60 seconds
        }

        print(f"Generating JWT assertion with payload: {payload} and header: {header}")
        # Convert JWK to RSA private key for signing

        # Helper function to decode base64url to int
        def b64url_to_int(b64url_str: str) -> int:
            """Convert base64url string to integer"""
            # Add padding if needed
            padding = 4 - len(b64url_str) % 4
            if padding != 4:
                b64url_str += '=' * padding
            # Replace URL-safe characters
            b64_str = b64url_str.replace('-', '+').replace('_', '/')
            decoded = base64.b64decode(b64_str)
            return int.from_bytes(decoded, byteorder='big')

        # Extract and decode key components from JWK
        n_int = b64url_to_int(private_jwk["n"])
        e_int = b64url_to_int(private_jwk["e"])
        d_int = b64url_to_int(private_jwk["d"])

        crt_fields = ("p", "q", "dp", "dq", "qi")
        has_all_crt_fields = all(private_jwk.get(field) for field in crt_fields)

        if has_all_crt_fields:
            p_int = b64url_to_int(private_jwk["p"])
            q_int = b64url_to_int(private_jwk["q"])
            dp_int = b64url_to_int(private_jwk["dp"])
            dq_int = b64url_to_int(private_jwk["dq"])
            qi_int = b64url_to_int(private_jwk["qi"])
        else:
            partial_crt_fields = [field for field in crt_fields if private_jwk.get(field)]
            if partial_crt_fields:
                raise ValueError(
                    "private_jwk must include either all CRT fields "
                    "('p', 'q', 'dp', 'dq', 'qi') or none of them."
                )

            p_int, q_int = rsa.rsa_recover_prime_factors(n_int, e_int, d_int)
            dp_int = rsa.rsa_crt_dmp1(d_int, p_int)
            dq_int = rsa.rsa_crt_dmq1(d_int, q_int)
            qi_int = rsa.rsa_crt_iqmp(p_int, q_int)

        private_numbers = rsa.RSAPrivateNumbers(
            p=p_int,
            q=q_int,
            d=d_int,
            dmp1=dp_int,
            dmq1=dq_int,
            iqmp=qi_int,
            public_numbers=rsa.RSAPublicNumbers(e_int, n_int)
        )

        private_key = private_numbers.private_key(backend=default_backend())

        # Encode JWT
        assertion = jwt.encode(
            payload,
            private_key,
            algorithm="RS256",
            headers=header
        )

        return assertion



def make_request(
    method: str,
    url: str,
    token: Optional[str] = None,
    data: Optional[Dict[str, Any]] = None,
    headers: Optional[Dict[str, str]] = None,
    timeout_seconds: int = 30,
    endpoint_name: str = "",
) -> requests.Response:
    """Make HTTP request to Okta API with timeout and response diagnostics."""
    req_headers: Dict[str, str] = {"Content-Type": "application/json"}
    if token:
        req_headers["Authorization"] = f"Bearer {token}"
    if headers:
        req_headers.update(headers)

    label = endpoint_name or url
    print(f"[HTTP] {method.upper()} {label}")
    print(req_headers)

    try:
        if method.upper() == "GET":
            response = requests.get(url, headers=req_headers, timeout=timeout_seconds)
        elif method.upper() == "POST":
            response = requests.post(url, json=data, headers=req_headers, timeout=timeout_seconds)
        elif method.upper() == "PUT":
            response = requests.put(url, json=data, headers=req_headers, timeout=timeout_seconds)
        else:
            raise ValueError(f"Unsupported method: {method}")
    except requests.Timeout as timeout_error:
        raise TimeoutError(f"Request timed out after {timeout_seconds}s: {label}") from timeout_error

    print(f"[HTTP] Status: {response.status_code}")
    if response.status_code >= 400:
        print(f"[HTTP] Error body: {response.text}")

    return response

def generate_rsa_key_pair(key_size: int = 2048) -> Tuple[Dict[str, str], Dict[str, str], str, str]:
    """Generate RSA key pair in JWK and PEM formats."""
    if not isinstance(key_size, int):
        raise TypeError("key_size must be an integer.")
    if key_size < 2048:
        raise ValueError("key_size must be at least 2048 bits.")
    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=key_size,
        backend=default_backend()
    )

    public_key = private_key.public_key()
    public_numbers = public_key.public_numbers()
    private_numbers = private_key.private_numbers()

    def int_to_base64url(value):
        byte_length = (value.bit_length() + 7) // 8
        value_bytes = value.to_bytes(byte_length, byteorder='big')
        return base64.urlsafe_b64encode(value_bytes).rstrip(b'=').decode('utf-8')

    key_id = str(uuid.uuid4())

    jwk_base = {
        "kty": "RSA",
        "kid": key_id,
        "use": "sig",
        "alg": "RS256",
        "n": int_to_base64url(public_numbers.n),
        "e": int_to_base64url(public_numbers.e)
    }

    public_jwk = dict(jwk_base)

    private_jwk = {
        **jwk_base,
        "d": int_to_base64url(private_numbers.d),
        "p": int_to_base64url(private_numbers.p),
        "q": int_to_base64url(private_numbers.q),
        "dp": int_to_base64url(private_numbers.dmp1),
        "dq": int_to_base64url(private_numbers.dmq1),
        "qi": int_to_base64url(private_numbers.iqmp)
    }

    private_pem = private_key.private_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
    ).decode('utf-8')

    return public_jwk, private_jwk, private_pem, key_id

print("Helper functions defined!")

---

## STEP 0: Generate RSA Key Pair

In [ ]:
print("Generating RSA Key Pair (2048-bit)...")
PUBLIC_JWK, PRIVATE_JWK, PRIVATE_PEM, KEY_ID = generate_rsa_key_pair(2048)

print("\n[SUCCESS] Key pair generated!")
print(f"\nKey ID: {KEY_ID}")
print(f"\nPublic JWK:")
print(json.dumps(PUBLIC_JWK, indent=2))

---

## STEP 1: Get Access Token

In [ ]:
print("Step 1: Getting Access Token...\n")

token_data = {
    'grant_type': 'client_credentials',
    'client_id': REG_SERVICE_CLIENT_ID,
    'client_assertion_type': 'urn:ietf:params:oauth:client-assertion-type:jwt-bearer',
    'client_assertion': _generate_jwt_assertion_for_service_token(REG_SERVICE_CLIENT_ID, f"{OKTA_DOMAIN}/oauth2/v1/token", REG_SERVICE_PRIVATE_JWK),
    'scope': 'okta.aiAgents.manage okta.aiAgents.read okta.governance.resourceOwner.manage okta.apps.manage okta.users.read okta.groups.read okta.authorizationServers.read'
}

resp = requests.post(
    f"{OKTA_DOMAIN}/oauth2/v1/token",
    data=token_data,
    headers={"Content-Type": "application/x-www-form-urlencoded"}
)

if resp.status_code == 200:
    access_token = resp.json()['access_token']
    print(f"[✓] Access token obtained - {access_token}")
else:
    print(f"[✗] Failed: {resp.status_code}")
    raise Exception("Failed to get access token")

---

## STEP 2: Get App ID

In [ ]:
print("Step 2: Getting App ID...\n")

resp = make_request(
    "GET",
    f"{OKTA_DOMAIN}/api/v1/apps?q={APP_NAME}",
    token=access_token,
    endpoint_name="List apps by query"
 )

if resp.status_code == 200:
    apps = resp.json()
    if not apps:
        raise Exception(f"No apps found for query '{APP_NAME}'")

    exact_matches = [app for app in apps if app.get("label") == APP_NAME or app.get("name") == APP_NAME]
    if len(exact_matches) == 1:
        app_id = exact_matches[0]["id"]
        print(f"[✓] Exact app match found! ID: {app_id}")
    elif len(exact_matches) > 1:
        print("[!] Multiple exact app matches found:")
        for app in exact_matches:
            print(f"    - {app.get('id')}: {app.get('label') or app.get('name')}")
        raise Exception("Multiple exact app matches found. Use a more specific APP_NAME.")
    else:
        app_id = apps[0]["id"]
        print("[!] No exact label/name match found; using first query result for training continuity.")
        print(f"    Selected app ID: {app_id}")
else:
    print(resp.text)
    raise Exception(f"Failed to fetch apps: {resp.status_code}")

---

## STEP 3: Create Agent

In [ ]:
print("Step 3: Creating AI Agent...\n")

agent_payload = {
    "appId": app_id,
    "profile": {
        "name": AGENT_NAME,
        "description": AGENT_DESCRIPTION
    }
}

resp = make_request(
    "POST",
    f"{OKTA_DOMAIN}/workload-principals/api/v1/ai-agents",
    token=access_token,
    data=agent_payload,
    endpoint_name="Create AI agent"
 )

if resp.status_code == 202:
    print(f"[✓] Agent creation request accepted (HTTP 202)")
    location = resp.headers.get('location')
    if location:
        operation_id = location.split('/')[-1]
        print(f"    Operation ID: {operation_id}")
    else:
        raise Exception("Create agent response did not include an operation location header.")
else:
    print(resp.text)
    raise Exception(f"Failed to create agent: {resp.status_code}")

---

## STEP 4: Get Agent ID from Operation

In [ ]:
print("Step 4: Getting Agent ID from Operation...\n")

max_attempts = 10
sleep_seconds = 2
agent_id = None

for attempt in range(1, max_attempts + 1):
    resp = make_request(
        "GET",
        f"{OKTA_DOMAIN}/workload-principals/api/v1/operations/{operation_id}",
        token=access_token,
        endpoint_name=f"Poll operation status (attempt {attempt}/{max_attempts})"
    )

    if resp.status_code != 200:
        print(resp.text)
        raise Exception(f"Failed to fetch operation status: {resp.status_code}")

    operation = resp.json()
    status = operation.get("status")
    print(f"    Attempt {attempt}: operation status = {status}")

    if status == "COMPLETED":
        agent_id = operation["resource"]["id"]
        print(f"[✓] Agent ID retrieved! {agent_id}")
        break

    if status in {"FAILED", "CANCELED"}:
        raise Exception(f"Operation finished with terminal status '{status}': {json.dumps(operation, indent=2)}")

    time.sleep(sleep_seconds)

if not agent_id:
    raise TimeoutError(f"Operation did not complete after {max_attempts} attempts.")

---

## STEP 5: Add Public Key to Agent

In [ ]:
print("Step 5: Adding Public Key to Agent...\n")

resp = make_request(
    "POST",
    f"{OKTA_DOMAIN}/workload-principals/api/v1/ai-agents/{agent_id}/credentials/jwks",
    token=access_token,
    data=PUBLIC_JWK
)

if resp.status_code == 201:
    key_response = resp.json()
    print(f"[✓] Public key added! Kid: {key_response['kid']}")
else:
    raise Exception(f"Failed: {resp.status_code}")

---

## STEP 6: Get Org ID

In [ ]:
print("Step 6: Getting Org ID...\n")

resp = make_request(
    "GET",
    f"{OKTA_DOMAIN}/.well-known/okta-organization",
    headers={"Accept": "application/json"}
)

if resp.status_code == 200:
    org_data = resp.json()
    org_id = org_data['id']
    print(f"[✓] Org ID: {org_id}")
else:
    raise Exception(f"Failed: {resp.status_code}")

---

## STEP 7: Get Owner Group ID

In [ ]:
print("Step 7: Getting Owner Group ID...\n")

OWNER_GROUP_NAME = "Beta Admins"
resp = make_request(
    "GET",
    f"{OKTA_DOMAIN}/api/v1/groups?q={OWNER_GROUP_NAME}",
    token=access_token,
    endpoint_name="List groups by query"
 )

owner_group_id = None
if resp.status_code == 200:
    groups = resp.json()
    exact_group_matches = [g for g in groups if g.get("profile", {}).get("name") == OWNER_GROUP_NAME]

    if len(exact_group_matches) == 1:
        owner_group_id = exact_group_matches[0]["id"]
        print(f"[✓] Owner group found! ID: {owner_group_id}")
    elif len(exact_group_matches) > 1:
        print("[!] Multiple exact owner group matches found:")
        for group in exact_group_matches:
            print(f"    - {group.get('id')}: {group.get('profile', {}).get('name')}")
        raise Exception("Multiple exact owner group matches found. Use a more specific group query.")
    else:
        message = f"No exact owner group match found for '{OWNER_GROUP_NAME}'."
        if STRICT_MODE:
            raise Exception(message)
        print(f"[!] {message}")
else:
    print(resp.text)
    if STRICT_MODE:
        raise Exception(f"Could not fetch groups: {resp.status_code}")
    print(f"[!] Could not fetch groups")

---

## STEP 8: Assign Owner to Agent

In [ ]:
print("Step 8: Assigning Owner to Agent...\n")

if owner_group_id:
    owner_payload = {
        "resourceOrns": [f"orn:okta:directory:{org_id}:workload-principals:ai-agents:{agent_id}"],
        "principalOrns": [f"orn:okta:directory:{org_id}:groups:{owner_group_id}"]
    }

    print(owner_payload)

    resp = make_request(
        "POST",
        f"{OKTA_DOMAIN}/governance/api/v1/resource-owners",
        token=access_token,
        data=owner_payload,
        endpoint_name="Assign owner to AI agent"
    )

    if resp.status_code in [200, 204]:
        print(f"[✓] Owner assigned!")
    else:
        print(resp.text)
        raise Exception(f"Owner assignment failed with status: {resp.status_code}")
else:
    message = "Skipping owner assignment because no owner group was resolved."
    if STRICT_MODE:
        raise Exception(message)
    print(f"[!] {message}")

---

## STEP 9: Activate Agent

In [ ]:
print("Step 9: Activating Agent...\n")

resp = make_request(
    "POST",
    f"{OKTA_DOMAIN}/workload-principals/api/v1/ai-agents/{agent_id}/lifecycle/activate",
    token=access_token,
    data={}
)

if resp.status_code == 202:
    print(f"[✓] Agent activated!")
else:
    raise Exception(f"Failed: {resp.status_code}")

---

## STEP 10: Add Managed Connection

In [ ]:
print("Step 10: Adding Managed Connection...\n")

okta_partition = get_okta_partition(OKTA_DOMAIN)
print(f"Using ORN partition derived from domain: {okta_partition}")

connection_payload = {
    "connectionType": "IDENTITY_ASSERTION_CUSTOM_AS",
    "authorizationServer": {
        "orn": f"orn:{okta_partition}:idp:{org_id}:authorization_servers:{AUTHZ_SERVER_ID}"
    },
    "scopeCondition": "INCLUDE_ONLY",
    "scopes": ["read_data"]
}

print(f"Payload: {json.dumps(connection_payload, indent=2)}\n")

resp = make_request(
    "POST",
    f"{OKTA_DOMAIN}/workload-principals/api/v1/ai-agents/{agent_id}/connections",
    token=access_token,
    data=connection_payload,
    endpoint_name="Create managed connection"
 )

if resp.status_code in [200, 201]:
    connection_response = resp.json()
    print(f"[✓] Managed connection created!")
    print(f"    Connection ID: {connection_response['id']}")
    print(f"    Connection Type: {connection_response['connectionType']}")
    print(f"    Scopes: {', '.join(connection_response.get('scopes', []))}")
else:
    print(f"[✗] Failed: {resp.status_code}")
    print(resp.text)
    raise Exception("Failed to create managed connection")

---

## Summary

In [ ]:
print("\n" + "="*80)
print("AI AGENT REGISTRATION COMPLETE")
print("="*80)
print(f"\n✓ Agent ID: {agent_id}")
print(f"✓ Agent Name: {AGENT_NAME}")
print(f"✓ Key: {PRIVATE_JWK}")
print(f"✓ Connection Type: IDENTITY_ASSERTION_CUSTOM_AS")
print(f"\n" + "="*80)